In [1]:
import asyncio
from pylabrobot.liquid_handling.backends.tecan.EVO_backend import EVOBackend

In [2]:
############### VARIABLES ###############

"""Shared Variables"""
z_travel = 2200

"""RoMa Variables"""
z_tray = 115
z_scanner = 1870
z_carousel = 470
z_sealer = 650
z_centrifuge = None
z_bucket_thin = 500
z_bucket_wide = 900
r_plate = 900
r_carousel = 2700
r_regrip = 1800
g_tray_open = 920
g_tray_closed = 765
g_bucket_open_thin_side = 800
g_bucket_open_thin_above = 1000
g_bucket_closed_thin = 720
g_bucket_open_wide = None
g_bucket_closed_wide = None

"""PnP Variables"""
z_decapper = 710
z_tiu = 620
z_rack = 570
z_bucket = None
g_small = 120
g_large = 160
g_open = 250
r_pick = 450
tube_column = [2755, 3005, 3255, 3505, 3755, 4005, 4255, 4505, 4755, 5005]
tube_row = [-30, 157, 345, 532, 720, 907, 1095, 1282, 1470, 1657, 1845, 2032, 2220, 2407, 2595, 2782]
bucket_column = [None, None, None, None, None, None]
bucket_row_1 = [None, None, None]
bucket_row_2 = [None, None, None]
bucket_row_3 = [None, None, None]
bucket_row_4 = [None, None, None]

"""Location Variables"""
class Point:
    def __init__(self, x, y, origin=None):
        self.x = x
        self.y = y
        self.origin = origin
    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"

plate_1 = Point(x=7125, y=405)
plate_2 = Point(x=7125, y=1360)
plate_3 = Point(x=7125, y=2310)
plate_4 = Point(x=8630, y=405)
plate_5 = Point(x=8630, y=1360)
plate_6 = Point(x=8630, y=2310)
sealer = Point(x=14080, y=3240)
carousel = Point(x=15850, y=2720)
scanner = Point(x=13910, y=-415)
bucket_1 = Point(x=13630, y=-30)
bucket_2 = Point(x=13630, y=930)
bucket_3 = Point(x=13630, y=1885)
bucket_4 = Point(x=13630, y=2880)
regrip = Point(x=15625, y=1900)
regrip_wide = Point(x=None, y=None)
centrifuge = Point(x=None, y=None)
decapper = Point(x=1260, y=-660)
tiu = Point(x=-120, y=1400)
tube_status = Point(x=None, y=None)
tube_rack = [
    [Point(x, y) for y in tube_row]
    for x in tube_column
    ]
bucket_rack_1 = [
    [Point(x, y) for y in bucket_row_1]
    for x in bucket_column
    ]
bucket_rack_2 = [
    [Point(x, y) for y in bucket_row_2]
    for x in bucket_column
    ]
bucket_rack_3 = [
    [Point(x, y) for y in bucket_row_3]
    for x in bucket_column
    ]
bucket_rack_4 = [
    [Point(x, y) for y in bucket_row_4]
    for x in bucket_column
    ]

In [3]:
############### BEGIN AND END FUNCTIONS ###############

"""""""""""""""""""""""""""""""""""""""""""""""""""
begin will establish USB connection, initialize the
FreedomEVO arms and move them to initial positions
"""""""""""""""""""""""""""""""""""""""""""""""""""
#PULNGER INITIALIZATION FOR LiHa should be implemented. TEST!!!
async def begin():
    backend = EVOBackend()
    await backend.io.setup()
    resp1 = await backend.send_command("W1", "PIA")
    print(resp1)
    resp2 = await backend.send_command("C5", "PIA")
    print(resp2)
    resp3 = await backend.send_command("C1", "PIA")
    print(resp3)
    resp4 = await backend.send_command("C5", "PID")
    print(resp4)
    await backend.send_command("C1", "PAA",[14628, 1999, 2000, 1800, 900])
    await backend.send_command("C5", "PAA", [9241, 793, 0, 1500, 1500, 1500, 1500, 1500, 1500, 1500, 1500])
    await backend.send_command("W1", "PAA", [-100, -700, 2000, 0, 280])

"""""""""""""""""""""""""""""""""""""""""""""""""""
end will return arms to initial positions and
terminate USB connection
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def end():
    await backend.send_command("C1", "PAA",[14628, 1999, 2000, 1800, 900])
    await backend.send_command("C5", "PAA", [9241, 793, 0, 1500, 1500, 1500, 1500, 1500, 1500, 1500, 1500])
    await backend.send_command("W1", "PAA", [-100, -700, 2000, 0, 280])
    await backend.stop

In [4]:
############### ROMA FUNCTIONS ###############

"""""""""""""""""""""""""""""""""""""""""""""""""""
The pick_up_tray function will take a plate location
as an argument and pick up the tray at that location
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def pick_up_tray(plate):
  await backend.send_command("C1", "PAZ", [z_travel])
  if plate == carousel:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_carousel, g_tray_open])
    await backend.send_command("C1", "PAZ", [z_carousel])
  elif plate == sealer:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_carousel, g_tray_open])
    await backend.send_command("C1", "PAZ", [z_sealer])
  elif plate == scanner:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_regrip, g_tray_open])
    await backend.send_command("C1", "PAZ", [z_scanner])
  else:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_plate, g_tray_open])
    await backend.send_command("C1", "PAZ", [z_tray])
  await backend.send_command("C1", "PAG", [g_tray_closed])
  await backend.send_command("C1", "PAZ", [z_travel])

"""""""""""""""""""""""""""""""""""""""""""""""""""
The transfer_tray_to function will take a plate location
as an argument and place a held tray at that location
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def transfer_tray_to(plate):
  await backend.send_command("C1", "PAZ", [z_travel])
  if plate == carousel:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_carousel, None])
    await backend.send_command("C1", "PAZ", [z_carousel])
  elif plate == sealer:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_carousel, None])
    await backend.send_command("C1", "PAZ", [z_sealer])
  elif plate == scanner:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_regrip, None])
    await backend.send_command("C1", "PAZ", [z_scanner])
  else:
    await backend.send_command("C1", "PAA", [plate.x, plate.y, None, r_plate, None])
    await backend.send_command("C1", "PAZ", [z_tray])
  await backend.send_command("C1", "PAG", [g_tray_open])
  await backend.send_command("C1", "PAZ", [z_travel])

"""""""""""""""""""""""""""""""""""""""""""""""""""
The move_tray function will take 2 arguments.
The first argument is a pickup location,
the second is where the tray will be deposited
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def move_tray(initial_location, end_location):
  await pick_up_tray(initial_location)
  await transfer_tray_to(end_location)

"""""""""""""""""""""""""""""""""""""""""""""""""""
The pick_up_bucket function will take a bucket location
as an argument and pick up the bucket at that location
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def pick_up_bucket(bucket):
  await backend.send_command("C1", "PAZ", [z_travel])
  if bucket == regrip:
    await backend.send_command("C1", "PAA", [bucket.x, bucket.y, None, r_regrip, g_bucket_open_thin_above])
    await backend.send_command("C1", "PAZ", [z_bucket_thin])
  else:
    await backend.send_command("C1", "PAA", [bucket.x + 1100 , bucket.y, None, r_plate, g_bucket_open_thin_side])
    await backend.send_command("C1", "PAZ", [z_bucket_thin])
    await backend.send_command("C1", "PAX", [bucket.x])
  await backend.send_command("C1", "PAG", [g_bucket_closed_thin])
  await backend.send_command("C1", "PAZ", [z_travel])

"""""""""""""""""""""""""""""""""""""""""""""""""""
The transfer_bucket_to function will take a bucket location
as an argument and place a held bucket at that location
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def transfer_bucket_to(bucket):
  await backend.send_command("C1", "PAZ", [z_travel])
  if bucket == regrip:
    await backend.send_command("C1", "PAA", [bucket.x, bucket.y, None, r_regrip, None])
    await backend.send_command("C1", "PAZ", [z_bucket_thin])
    await backend.send_command("C1", "PAG", [g_bucket_open_thin_above])
  else:
    await backend.send_command("C1", "PAA", [bucket.x, bucket.y, None, r_plate, None])
    await backend.send_command("C1", "PAZ", [z_bucket_thin])
    await backend.send_command("C1", "PAG", [g_bucket_open_thin_side])
    await backend.send_command("C1", "PAX", [bucket.x + 1100])
  await backend.send_command("C1", "PAZ", [z_travel])

"""""""""""""""""""""""""""""""""""""""""""""""""""
The move_bucket function will take 2 arguments.
The first argument is a pickup location,
the second is where the tray will be deposited
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def move_bucket(initial_location, end_location):
  await pick_up_bucket(initial_location)
  await transfer_bucket_to(end_location)

"""""""""""""""""""""""""""""""""""""""""""""""""""
The move_into_centrifuge function will transfer a bucket
from the regrip plate into the centrifuge
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def move_into_centrifuge():
  await backend.send_command("C1", "PAZ", [z_travel])
  await backend.send_command("C1", "PAA", [regrip_wide.x, regrip_wide.y, None, r_plate, g_bucket_open_wide])
  await backend.send_command("C1", "PAZ", [z_bucket_wide])
  await backend.send_command("C1", "PAG", [g_bucket_closed_wide])
  await backend.send_command("C1", "PAZ", [z_travel])
  await backend.send_command("C1", "PAA", [centrifuge.x, centrifuge.y, None, r_regrip, None])
  await backend.send_command("C1", "PAZ", [z_centrifuge])
  await backend.send_command("C1", "PAG", [g_bucket_open_wide])
  await backend.send_command("C1", "PAZ", [z_travel])

"""""""""""""""""""""""""""""""""""""""""""""""""""
The move_from_centrifuge function will transfer a bucket
from the centrifuge onto the regrip plate
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def move_from_centrifuge():
  await backend.send_command("C1", "PAZ", [z_travel])
  await backend.send_command("C1", "PAA", [centrifuge.x, centrifuge.y, None, r_regrip, g_bucket_open_wide])
  await backend.send_command("C1", "PAZ", [z_centrifuge])
  await backend.send_command("C1", "PAG", [g_bucket_closed_wide])
  await backend.send_command("C1", "PAZ", [z_travel])
  await backend.send_command("C1", "PAA", [regrip_wide.x, regrip_wide.y, None, r_plate, None])
  await backend.send_command("C1", "PAZ", [z_bucket_wide])
  await backend.send_command("C1", "PAG", [g_bucket_open_wide])
  await backend.send_command("C1", "PAZ", [z_travel])


In [5]:
############### LIHA FUNCTIONS ###############

In [6]:
############### PNP FUNCTIONS ###############

"""""""""""""""""""""""""""""""""""""""""""""""""""
The pick_up_tube function will take a tube location
as an argument and pick up the tube at that location
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def pick_up_tube(location):
  await backend.send_command("W1", "PAZ", [z_travel])
  await backend.send_command("W1", "PAA", [location.x, location.y, None, r_pick, g_open])

  if location == decapper:
    await backend.send_command("W1", "PAZ", [z_decapper])
    if location.origin.x < 4255:
      await backend.send_command("W1", "PAG", [g_large])
    else:
      await backend.send_command("W1", "PAG", [g_small])
    tube_status.origin = location.origin

  elif location == tiu:
    await backend.send_command("W1", "PAZ", [z_tiu])
    if location.origin.x < 4255:
      await backend.send_command("W1", "PAG", [g_large])
    else:
      await backend.send_command("W1", "PAG", [g_small])
    tube_status.origin = location.origin

  elif any(location in rack for rack in (bucket_rack_1, bucket_rack_2, bucket_rack_3, bucket_rack_4)):
    await backend.send_command("W1", "PAR", [0])
    await backend.send_command("W1", "PAZ", [z_bucket])
    if location.origin.x < 4255:
      await backend.send_command("W1", "PAG", [g_large])
    else:
      await backend.send_command("W1", "PAG", [g_small])
    tube_status.origin = location.origin

  else:
    await backend.send_command("W1", "PAZ", [z_rack])
    if location.x < 4255:
      await backend.send_command("W1", "PAG", [g_large])
    else:
      await backend.send_command("W1", "PAG", [g_small])
    tube_status.origin = location

  await backend.send_command("W1", "PAZ", [z_travel])

"""""""""""""""""""""""""""""""""""""""""""""""""""
The transfer_tube_to function will take a tube location
as an argument and place a held tube at that location
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def transfer_tube_to(location):
  await backend.send_command("W1", "PAZ", [z_travel])

  if location == decapper:
    await backend.send_command("W1", "PAA", [location.x, location.y, None, None, None])
    await backend.send_command("W1", "PAZ", [z_decapper])
    await backend.send_command("W1", "PAG", [g_open])
    location.origin = tube_status.origin

  elif location == tiu:
    await backend.send_command("W1", "PAA", [location.x, location.y, None, None, None])
    await backend.send_command("W1", "PAZ", [z_tiu])
    await backend.send_command("W1", "PAG", [g_open])
    location.origin = tube_status.origin

  elif any(location in rack for rack in (bucket_rack_1, bucket_rack_2, bucket_rack_3, bucket_rack_4)):
    await backend.send_command("W1", "PAA", [location.x, location.y, None, 0, None])
    await backend.send_command("W1", "PAZ", [z_bucket])
    await backend.send_command("W1", "PAG", [g_open])
    location.origin = tube_status.origin

  else:
    await backend.send_command("W1", "PAA", [tube_status.origin.x, tube_status.origin.y, None, r_pick, None])
    await backend.send_command("W1", "PAZ", [z_rack])
    await backend.send_command("W1", "PAG", [g_open])

  await backend.send_command("W1", "PAZ", [z_travel])

"""""""""""""""""""""""""""""""""""""""""""""""""""
The move_tube function will take 2 arguments.
The first argument is a pickup location,
the second is where the tube will be deposited
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def move_tube(initial_location, end_location):
  await pick_up_tube(initial_location)
  await transfer_tube_to(end_location)


In [7]:
############### TEST FUNCTIONS ###############

"""""""""""""""""""""""""""""""""""""""""""""""""""
The roma_tray function will run the RoMa through all
the motions required to process a tray.
***BE SURE TO MOVE PNP AND LIHA TO THE FAR LEFT BEFORE RUNNING***
"""""""""""""""""""""""""""""""""""""""""""""""""""
async def roma_tray():
  await backend.send_command("C1", "PIA")
  await move_tray(carousel, scanner)
  await move_tray(scanner, plate_1)
  await move_tray(carousel, scanner)
  await move_tray(scanner, plate_2)
  await move_tray(carousel, scanner)
  await move_tray(scanner, plate_3)
  await move_tray(carousel, scanner)
  await move_tray(scanner, plate_4)
  await move_tray(carousel, scanner)
  await move_tray(scanner, plate_5)
  await move_tray(carousel, scanner)
  await move_tray(scanner, plate_6)
  await move_tray(plate_1, sealer)
  await move_tray(sealer, carousel)
  await move_tray(plate_2, sealer)
  await move_tray(sealer, carousel)
  await move_tray(plate_3, sealer)
  await move_tray(sealer, carousel)
  await move_tray(plate_4, sealer)
  await move_tray(sealer, carousel)
  await move_tray(plate_5, sealer)
  await move_tray(sealer, carousel)
  await move_tray(plate_6, sealer)
  await move_tray(sealer, carousel)
  await backend.send_command("C1", "PAA", [4500, 450, None, 0, None])

In [8]:
############### TESTING CELL ###############

backend = EVOBackend()
await backend.io.setup()

#await backend.send_command("W1", "PIA")
#await backend.send_command("W1", "PAA", [1260, -660,660,0, g_small])
#await backend.send_command("W1", "PAR", [0])
await move_tube(tube_rack[1][1], decapper)
await move_tube(tube_rack[9][15], tiu)
await move_tube(decapper, tube_rack)
await move_tube(tiu, tube_rack)


backend.stop()

<coroutine object TecanLiquidHandler.stop at 0x0000014192109FC0>